In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTENC
from imblearn.under_sampling import TomekLinks
import pandas as pd

In [2]:
# Load both datasets
df = pd.read_csv("C:\\Users\\VP678WV\\OneDrive - EY\\Documents\\Delivery_Delay\\data\\processed\\new_engineered_features2.csv")
# Preview the result
# print(df.head())

In [3]:
df["label"].value_counts()

label
 1    8909
-1    3714
 0    2925
Name: count, dtype: int64

In [4]:
df['order_full_location'] = (
    df['order_region'].astype(str) + "|" +
    df['order_country'].astype(str) + "|" +
    df['order_state'].astype(str) + "|" +
    df['order_city'].astype(str)
)

df['customer_full_location'] = (
    df['customer_country'].astype(str) + "|" +
    df['customer_state'].astype(str) + "|" +
    df['customer_city'].astype(str)
)

STATISTICAL ANALYSIS ON CUSTOMER AND ORDER LOCATIONS

In [5]:
import pandas as pd
import scipy.stats as stats
from scipy.stats import pearsonr, spearmanr

# Columns you want to group by
group_cols = [
    'order_full_location', 'customer_full_location',
    'order_dayofweek', 'shipping_dayofweek', 
    'order_hour', 'shipping_hour', 
    'order_daynight', 'ship_daynight'
]

# Weights for performance score
w_delay = 1.0
w_early = 0.5
w_ontime = 1.0

for col in group_cols:
    print(f"\n===== Analysis for {col} =====")
    
    # Step 1: Aggregate from TRAIN only
    df_grouped = df.groupby(col).agg(
        order_count=('label', 'size'),
        delay_pct=('label', lambda x: (x == 1).mean() * 100),
        early_pct=('label', lambda x: (x == -1).mean() * 100),
        ontime_pct=('label', lambda x: (x == 0).mean() * 100)
    ).reset_index()
    
    # Step 2: Performance score
    df_grouped['performance_score'] = (
        w_delay * df_grouped['delay_pct'] +
        w_early * df_grouped['early_pct'] -
        w_ontime * df_grouped['ontime_pct']
    )
    
    # Step 3: Category bins
    df_grouped['performance_category'] = pd.cut(
        df_grouped['performance_score'],
        bins=[-101, -75, -25, 25, 75, 100],
        labels=['very low delay', 'low delay', 'medium delay', 'high delay', 'extreme delay']
    )

    # Step 4: Always rename to have suffix
    score_col = f'performance_score_{col}'
    cat_col = f'performance_category_{col}'
    df_grouped.rename(columns={
        'performance_score': score_col,
        'performance_category': cat_col
    }, inplace=True)

    # print(df_grouped)

    # Step 5: Merge into TRAIN
    df = df.merge(
        df_grouped[[col, score_col, cat_col]],
        on=col,
        how='left'
    )


    # Step 7: ANOVA test (train only)
    group_early = df[df['label'] == -1][score_col].dropna()
    group_ontime = df[df['label'] == 0][score_col].dropna()
    group_delay = df[df['label'] == 1][score_col].dropna()

    f_stat, p_value = stats.f_oneway(group_early, group_ontime, group_delay)
    print(f"ANOVA F-statistic: {f_stat:.4f}, p-value: {p_value:.4e} --> {'Good to have' if p_value < 0.05 else 'Not good to have'}")

    # Step 8: Chi-square test
    contingency_table = pd.crosstab(df[cat_col], df['label'])
    chi2, p, dof, expected = stats.chi2_contingency(contingency_table)
    print(f"Chi-square: {chi2:.4f}, p-value: {p:.4e} --> {'Good to have' if p < 0.05 else 'Not good to have'}")

    # Step 9: Correlation
    pearson_corr, pearson_p = pearsonr(df[score_col], df['label'])
    spearman_corr, spearman_p = spearmanr(df[score_col], df['label'])

    print(f"Pearson corr: {pearson_corr:.4f}, p-value: {pearson_p:.4e} --> {'Good to have' if pearson_p < 0.05 else 'Not good to have'}")
    print(f"Spearman corr: {spearman_corr:.4f}, p-value: {spearman_p:.4e} --> {'Good to have' if spearman_p < 0.05 else 'Not good to have'}")



===== Analysis for order_full_location =====
ANOVA F-statistic: 1673.3867, p-value: 0.0000e+00 --> Good to have
Chi-square: 2637.5104, p-value: 0.0000e+00 --> Good to have
Pearson corr: 0.1940, p-value: 1.1530e-131 --> Good to have
Spearman corr: 0.2255, p-value: 1.9364e-178 --> Good to have

===== Analysis for customer_full_location =====
ANOVA F-statistic: 289.8970, p-value: 2.4572e-124 --> Good to have
Chi-square: 387.5930, p-value: 8.4311e-79 --> Good to have
Pearson corr: 0.0868, p-value: 2.0279e-27 --> Good to have
Spearman corr: 0.0893, p-value: 6.7941e-29 --> Good to have

===== Analysis for order_dayofweek =====
ANOVA F-statistic: 5.6885, p-value: 3.3918e-03 --> Good to have
Chi-square: 0.0000, p-value: 1.0000e+00 --> Not good to have
Pearson corr: -0.0067, p-value: 4.0260e-01 --> Not good to have
Spearman corr: -0.0031, p-value: 7.0309e-01 --> Not good to have

===== Analysis for shipping_dayofweek =====
ANOVA F-statistic: 3.2441, p-value: 3.9028e-02 --> Good to have
Chi-squ

In [6]:
import pandas as pd

# Dictionary to store all score DataFrames
performance_dict = {}

group_cols = [
    'order_full_location', 'customer_full_location',
    'order_dayofweek', 'shipping_dayofweek',
    'order_hour', 'shipping_hour',
    'order_daynight', 'ship_daynight'
]

w_delay, w_early, w_ontime = 1.0, 0.5, 1.0

for col in group_cols:
    df_grouped = df.groupby(col).agg(
        order_count=('label', 'size'),
        delay_pct=('label', lambda x: (x == 1).mean() * 100),
        early_pct=('label', lambda x: (x == -1).mean() * 100),
        ontime_pct=('label', lambda x: (x == 0).mean() * 100)
    ).reset_index()

    # Compute score
    df_grouped['performance_score'] = (
        w_delay * df_grouped['delay_pct'] +
        w_early * df_grouped['early_pct'] -
        w_ontime * df_grouped['ontime_pct']
    )

    # Category
    df_grouped['performance_category'] = pd.cut(
        df_grouped['performance_score'],
        bins=[-101, -75, -25, 25, 75, 100],
        labels=['very low delay', 'low delay', 'medium delay', 'high delay', 'extreme delay']
    )

    # Rename cols to avoid clashes
    df_grouped.rename(columns={
        'performance_score': f'performance_score_{col}',
        'performance_category': f'performance_category_{col}'
    }, inplace=True)

    # Save in dictionary
    performance_dict[col] = df_grouped

# Save to Excel (each col in separate sheet)
with pd.ExcelWriter("performance_scores.xlsx") as writer:
    for col, df_grouped in performance_dict.items():
        df_grouped.to_excel(writer, sheet_name=col, index=False)

print("✅ All performance scores saved to performance_scores.xlsx")

✅ All performance scores saved to performance_scores.xlsx


In [7]:
len(df.columns)

50

In [8]:
# Drop performance_category_* and source columns
drop_cols = [col for col in df.columns if col.startswith("performance_category_")]
print(f"Dropping {len(drop_cols)} performance_category columns: {drop_cols}")
drop_cols.extend(['order_full_location', 'customer_full_location', 
                        'order_dayofweek', 'shipping_dayofweek', 
                        'order_hour', 'shipping_hour', 
                        'order_daynight', 'ship_daynight',
                        'order_city', 'order_state', 'order_country', 'order_region', 'order_full_location',
    'customer_city', 'customer_state', 'customer_country', 'customer_full_location',])

print(f"Dropping {len(drop_cols)} source columns: {drop_cols}")

df = df.drop(columns=drop_cols)

Dropping 8 performance_category columns: ['performance_category_order_full_location', 'performance_category_customer_full_location', 'performance_category_order_dayofweek', 'performance_category_shipping_dayofweek', 'performance_category_order_hour', 'performance_category_shipping_hour', 'performance_category_order_daynight', 'performance_category_ship_daynight']
Dropping 25 source columns: ['performance_category_order_full_location', 'performance_category_customer_full_location', 'performance_category_order_dayofweek', 'performance_category_shipping_dayofweek', 'performance_category_order_hour', 'performance_category_shipping_hour', 'performance_category_order_daynight', 'performance_category_ship_daynight', 'order_full_location', 'customer_full_location', 'order_dayofweek', 'shipping_dayofweek', 'order_hour', 'shipping_hour', 'order_daynight', 'ship_daynight', 'order_city', 'order_state', 'order_country', 'order_region', 'order_full_location', 'customer_city', 'customer_state', 'cust

In [9]:
df.shape

(15548, 27)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15548 entries, 0 to 15547
Data columns (total 27 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   payment_type                              15548 non-null  object 
 1   profit_per_order                          15548 non-null  float64
 2   category_name                             15548 non-null  object 
 3   customer_segment                          15548 non-null  object 
 4   market                                    15548 non-null  object 
 5   order_item_discount                       15548 non-null  float64
 6   order_item_product_price                  15548 non-null  float64
 7   order_item_profit_ratio                   15548 non-null  float64
 8   order_item_quantity                       15548 non-null  float64
 9   sales                                     15548 non-null  float64
 10  order_profit_per_order            

BALANCING DATASET

In [11]:
# --------------------------------
# Step 1: Prepare data
# --------------------------------
# X = features, y = target
X = df.drop('label', axis=1)
y = df['label']

# Get their column indices
# cat_indices = [X.columns.get_loc(col) for col in categorical_cols]

In [12]:
# --------------------------------
# Step 2: Split before sampling
# --------------------------------
x_train, x_test, y_train, y_test = train_test_split(
    X, y,
    stratify=y,
    test_size=0.2,
    random_state=42
)

In [ ]:
# cols_to_convert = ['order_to_shipment_days', 'order_to_shipment_planned_days',
#        'shipment_delay_days']  # columns you want as int
# df[cols_to_convert] = df[cols_to_convert].astype(int)

In [13]:
# # 1. Encode first
categorical_cols = [
    'payment_type', 'category_name', 'customer_segment',
    'market', 'product_name', 'shipping_mode',
    'order_to_shipment_days',
    'order_to_shipment_planned_days', 'shipment_delay_days'
]

# Encode all object/string columns
# categorical_cols = x_train.select_dtypes(include=['object']).columns

le_dict = {}
for col in categorical_cols:
    le = LabelEncoder()
    x_train[col] = le.fit_transform(x_train[col])
    le_dict[col] = le  # store encoders if needed later

In [14]:
cat_indices = [x_train.columns.get_loc(col) for col in categorical_cols]

In [15]:
# # Assuming your label column name is 'label' (change if needed)
# label_col = 'label'

# # Separate features and target
# y_train = x_train[label_col]
# x_train = x_train.drop(columns=[label_col])

# Print initial sizes
print(f"Initial Train Size: {x_train.shape}, Test Size: {x_test.shape}")
print("Class balance in y_train:\n", y_train.value_counts(), "\n")

# --------------------------------
# Step 3: SMOTENC oversampling
# --------------------------------
# Only oversample specific classes (example: 0 and -1 to 5000 each)
smote_nc = SMOTENC(
    categorical_features=cat_indices,
    sampling_strategy={0: 5000, -1: 5000},  # target counts
    random_state=42
)
x_train_over, y_train_over = smote_nc.fit_resample(x_train, y_train)
print(f"After SMOTENC - Train Size: {x_train_over.shape}")
print("Class balance after SMOTENC:\n", pd.Series(y_train_over).value_counts(), "\n")

# --------------------------------
# Step 4: Tomek Links undersampling
# --------------------------------
tl = TomekLinks(sampling_strategy='auto')
x_train_bal, y_train_bal = tl.fit_resample(x_train_over, y_train_over)
print(f"After TomekLinks - Train Size: {x_train_bal.shape}")
print("Class balance after SMOTENC + TomekLinks:\n", pd.Series(y_train_bal).value_counts(), "\n")

# --------------------------------
# Step 5: Final check
# --------------------------------
print(f"Final Test Size (unchanged): {x_test.shape}")
print("Final Train Size:", x_train_bal.shape)

Initial Train Size: (12438, 26), Test Size: (3110, 26)
Class balance in y_train:
 label
 1    7127
-1    2971
 0    2340
Name: count, dtype: int64 

After SMOTENC - Train Size: (17127, 26)
Class balance after SMOTENC:
 label
 1    7127
-1    5000
 0    5000
Name: count, dtype: int64 

After TomekLinks - Train Size: (16033, 26)
Class balance after SMOTENC + TomekLinks:
 label
 1    6377
-1    5000
 0    4656
Name: count, dtype: int64 

Final Test Size (unchanged): (3110, 26)
Final Train Size: (16033, 26)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def compare_feature_distribution(X_before, X_after, feature_name):
    plt.figure(figsize=(8,5))
    sns.kdeplot(X_before[feature_name], label='Before', fill=True, alpha=0.5)
    sns.kdeplot(X_after[feature_name], label='After', fill=True, alpha=0.5)
    plt.title(f"Distribution of {feature_name} Before vs After Resampling")
    plt.legend()
    plt.show()

def compare_categorical_distribution(X_before, X_after, feature_name):
    plt.figure(figsize=(8,5))
    sns.countplot(x=X_before[feature_name], color='blue', alpha=0.5, label='Before')
    sns.countplot(x=X_after[feature_name], color='orange', alpha=0.5, label='After')
    plt.title(f"Categorical Distribution of {feature_name} Before vs After Resampling")
    plt.legend()
    plt.show()

In [ ]:
num_features = x_train.select_dtypes(include=['int64','float64']).columns
cat_features = categorical_cols  # already defined earlier

for col in num_features:
    compare_feature_distribution(x_train, x_train_bal, col)

for col in cat_features:
    compare_categorical_distribution(x_train, x_train_bal, col)

In [16]:
x_train_bal

,payment_type,profit_per_order,category_name,customer_segment,market,order_item_discount,order_item_product_price,order_item_profit_ratio,order_item_quantity,sales,...,order_to_shipment_planned_days,shipment_delay_days,performance_score_order_full_location,performance_score_customer_full_location,performance_score_order_dayofweek,performance_score_shipping_dayofweek,performance_score_order_hour,performance_score_shipping_hour,performance_score_order_daynight,performance_score_ship_daynight
0,0,10.314638,30,2,2,37.485000,49.980000,0.050000,5.000000,249.900000,...,3,1,45.515695,48.733738,48.784799,48.462214,51.800327,51.818182,53.411233,53.140187
1,2,-4.472153,47,0,1,13.000000,50.000000,0.052072,2.000000,100.000000,...,3,2,100.000000,92.857143,48.784799,51.288210,45.763994,45.392749,50.158793,49.980119
2,0,81.892784,30,0,2,32.487000,49.980000,0.330000,5.000000,249.900000,...,2,3,44.791667,48.733738,50.497512,48.889917,49.849398,50.540958,51.157935,51.416539
3,1,134.113050,9,0,2,47.996800,299.980000,0.500000,1.000000,299.980000,...,1,4,100.000000,39.285714,50.913242,51.909802,54.545455,54.413893,53.411233,53.140187
4,0,22.352833,12,0,2,29.995000,59.990000,0.310000,2.000000,119.980000,...,1,4,42.424242,56.097561,47.946751,50.656703,54.545455,54.413893,53.411233,53.140187
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16028,1,47.510338,12,0,1,20.462551,59.990000,0.298901,3.000000,179.970000,...,2,3,23.229386,59.674489,50.866964,51.448064,54.407471,54.305203,52.701420,52.597220
16029,0,45.687490,30,0,3,2.897250,49.980000,0.495508,2.000000,99.960000,...,3,1,53.361422,48.733738,51.060177,51.770191,50.659140,50.964795,50.696403,50.968000
16030,3,49.636890,12,0,2,2.643007,59.297773,0.270000,3.069154,181.349614,...,3,5,0.740931,44.741064,48.296386,52.386678,51.627356,51.703321,53.113307,52.882888
16031,1,38.479750,38,0,1,6.140836,27.679735,0.395525,3.345003,88.555797,...,3,6,67.508315,60.125302,49.151958,48.816138,57.501105,56.646579,52.930101,52.648108


In [17]:
# --------------------------------
# Step 5: Decode categorical columns back to original values
# --------------------------------
# Convert to DataFrame to restore column names
X_train_bal_org = pd.DataFrame(x_train_bal, columns=x_train.columns)
X_test_org = pd.DataFrame(x_test, columns=x_test.columns)

for col in categorical_cols:
    le = le_dict[col]
    X_train_bal_org[col] = le.inverse_transform(X_train_bal_org[col].astype(int))
    # X_test_org[col] = le.inverse_transform(X_test_org[col].astype(int))

In [18]:
X_train_bal_org.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16033 entries, 0 to 16032
Data columns (total 26 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   payment_type                              16033 non-null  object 
 1   profit_per_order                          16033 non-null  float64
 2   category_name                             16033 non-null  object 
 3   customer_segment                          16033 non-null  object 
 4   market                                    16033 non-null  object 
 5   order_item_discount                       16033 non-null  float64
 6   order_item_product_price                  16033 non-null  float64
 7   order_item_profit_ratio                   16033 non-null  float64
 8   order_item_quantity                       16033 non-null  float64
 9   sales                                     16033 non-null  float64
 10  order_profit_per_order            

In [19]:
X_test_org.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3110 entries, 7031 to 2399
Data columns (total 26 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   payment_type                              3110 non-null   object 
 1   profit_per_order                          3110 non-null   float64
 2   category_name                             3110 non-null   object 
 3   customer_segment                          3110 non-null   object 
 4   market                                    3110 non-null   object 
 5   order_item_discount                       3110 non-null   float64
 6   order_item_product_price                  3110 non-null   float64
 7   order_item_profit_ratio                   3110 non-null   float64
 8   order_item_quantity                       3110 non-null   float64
 9   sales                                     3110 non-null   float64
 10  order_profit_per_order                

In [20]:
for col in X_train_bal_org.columns:
    print(f"Column: {col}  |  dtype: {x_train[col].dtype}")
    print(f"Unique values ({x_train[col].nunique()}): {x_train[col].unique()[:10]}")
    print("-" * 50)

Column: payment_type  |  dtype: int64
Unique values (4): [0 2 1 3]
--------------------------------------------------
Column: profit_per_order  |  dtype: float64
Unique values (12095): [  10.314638   -4.472153   81.892784  134.11305    22.352833   11.252522
 -150.03581    13.821517  119.987854   59.451817]
--------------------------------------------------
Column: category_name  |  dtype: int64
Unique values (50): [30 47  9 12 38 11 46 34 18 24]
--------------------------------------------------
Column: customer_segment  |  dtype: int64
Unique values (3): [2 0 1]
--------------------------------------------------
Column: market  |  dtype: int64
Unique values (5): [2 1 4 0 3]
--------------------------------------------------
Column: order_item_discount  |  dtype: float64
Unique values (1008): [37.485   13.      32.487   47.9968  29.995    6.       2.7993  15.996
 67.18185  7.497  ]
--------------------------------------------------
Column: order_item_product_price  |  dtype: float64
Un

In [21]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

numerical_features_to_scale = [
    "profit_per_order",
    "order_item_discount",
    "order_item_product_price",
    "order_item_profit_ratio",
    "order_item_quantity",  # num discrete
    "sales",
    "order_profit_per_order",
    "distance_normalized",
    "order_shipping_time",
    "order_to_shipment_days",
    "order_to_shipment_planned_days",
    "shipment_delay_days",
    # Newly added performance score columns
    "performance_score_order_full_location",
    "performance_score_customer_full_location",
    "performance_score_order_dayofweek",
    "performance_score_shipping_dayofweek",
    "performance_score_order_hour",
    "performance_score_shipping_hour",
    "performance_score_order_daynight",
    "performance_score_ship_daynight"
]


# Fit only on training data
scaler.fit(X_train_bal_org[numerical_features_to_scale])

# Transform both
X_train_bal_org[numerical_features_to_scale] = scaler.transform(
    X_train_bal_org[numerical_features_to_scale]
)
X_test_org[numerical_features_to_scale] = scaler.transform(
    X_test_org[numerical_features_to_scale]
)

In [22]:
import joblib

# Save the trained scaler
joblib.dump(scaler, "numerical_scaler.pkl")

['numerical_scaler.pkl']

In [23]:
X_test_org.shape

(3110, 26)

In [24]:
X_train_bal_org.shape

(16033, 26)

In [25]:
import pandas as pd

categorical_cols = [
    'payment_type', 'category_name', 
    'customer_segment', 'market', 'product_name', 'shipping_mode'
]

# Count unique values for each categorical column
unique_counts = {col: df[col].nunique() for col in categorical_cols}

# Display nicely
for col, count in unique_counts.items():
    print(f"{col}: {count} unique values")

payment_type: 4 unique values
category_name: 50 unique values
customer_segment: 3 unique values
market: 5 unique values
product_name: 116 unique values
shipping_mode: 4 unique values


In [26]:
# ---------------------------------------------------------------------
# Removing 'category_name' and 'product_name'
# Reason:
# These are product identifiers / category labels that don't directly
# cause shipping delay. Their effects are already captured through
# numerical features like 'order_item_price', 'discount', 'quantity',
# and other handling-related attributes.
# Keeping them would introduce redundancy and potentially increase
# dimensionality without adding new information.
# ---------------------------------------------------------------------
X_train_bal_org = X_train_bal_org.drop(columns=['category_name', 'product_name'])
X_test_org = X_test_org.drop(columns=['category_name', 'product_name'])

In [27]:
X_train_bal_org = X_train_bal_org.drop(columns=['market', 'customer_segment'])
X_test_org = X_test_org.drop(columns=['market', 'customer_segment'])

In [28]:
shipping_map = {
    'Standard Class': 1,
    'Second Class': 2,
    'First Class': 3,
    'Same Day': 4
}
X_train_bal_org['shipping_mode'] = X_train_bal_org['shipping_mode'].map(shipping_map)
X_test_org['shipping_mode'] = X_test_org['shipping_mode'].map(shipping_map)

In [29]:
import pandas as pd

ohe_cols = ['payment_type']
X_train_bal_org = pd.get_dummies(X_train_bal_org, columns=ohe_cols, drop_first=False)  # keep all for interpretability
X_test_org = pd.get_dummies(X_test_org, columns=ohe_cols, drop_first=False) 

In [40]:
y_test.info()

<class 'pandas.core.series.Series'>
Index: 3110 entries, 7031 to 2399
Series name: label
Non-Null Count  Dtype
--------------  -----
3110 non-null   int64
dtypes: int64(1)
memory usage: 48.6 KB


In [41]:
# Assuming your target variables are named y_train_bal and y_test

# Combine X and y for train
train_combined = X_train_bal_org.copy()
train_combined['target'] = y_train_bal  # replace 'target' with actual column name

# Combine X and y for test
test_combined = X_test_org.copy()
test_combined['target'] = y_test  # replace 'target' with actual column name

# Export to CSV
train_combined.to_csv("train_encoded3.csv", index=False)
test_combined.to_csv("test_encoded3.csv", index=False)